# search in action

`search` is the *what it means* verb: text in, one ranked list of entries out. This notebook builds a small
namespace from this repository's own files, indexes it, and walks through what a search result carries —
the ranked rows, the chunk regions with their line bounds, the query-biased previews, scoping, the
freshness overlay, and the term statistics the ranking used.

Today search is lexical-only (BM25 over block postings in the Rust engine). The vector leg and fusion are
the next specs; the verb, the result shape, and the renderer stay the same when they land.

Run the cells top to bottom. Everything is local: a throwaway sqlite file in a temp directory.

In [1]:
import tempfile
from pathlib import Path as FsPath

from vfs import AsyncVirtualFileSystem
from vfs.models import Entry
from vfs.paths import Path
from vfs.results.preview import select_preview
from vfs.storage.backends import DatabaseStorage

# Find the repository root from wherever this notebook is opened.
root = FsPath.cwd()
while not (root / "pyproject.toml").exists():
    root = root.parent
workdir = FsPath(tempfile.mkdtemp())
print("repo:", root)
print("scratch:", workdir)

repo: /Users/claygendron/Git/Repos/vfs
scratch: /var/folders/_c/n40__q_16zs0cbn45nb3c_nw0000gn/T/tmpy3c8p247


## 1. Build a namespace

Files become `Entry` rows. We take the library's own source and the research memos — code and prose in one corpus,
about 200 files. `parents=True` creates the directories along the way.

In [2]:
def entries_under(folder: str, pattern: str) -> list[Entry]:
    out = []
    for file in sorted((root / folder).rglob(pattern)):
        if "__pycache__" in file.parts:
            continue
        out.append(Entry(path=Path("/" + file.relative_to(root).as_posix()), content=file.read_text()))
    return out

corpus = entries_under("src/vfs", "*.py") + entries_under("context/research", "*.md")
storage = DatabaseStorage(url=f"sqlite+aiosqlite:///{workdir}/search.sqlite")
written = await storage.write(entries=corpus, parents=True)
print(written.success, len(corpus), "files written")

True 557 files written


## 2. Index it

`rebuild()` splits each file into chunks, builds the gram index grep uses, and builds the lexical index search uses.
Until it runs, search still answers — from the *overlay*, the live text of unindexed files (section 6).

In [3]:
indexed = await storage.rebuild()
print(indexed.success)
print(indexed.to_str())

True
INFO: Not found: 11360 markdown references across 462 documents name no live entry — Check the path with ls or glob, or create the entry first. (retry: never)


## 3. The first search

The rendered text is what an agent reads. Rows come back **in rank order** — never sorted by path. Each entry prints a
header with its score, then each of its best chunk regions as a `path:start-end` locator with a short preview
quoted beneath it. Query terms are bolded in Markdown.

In [4]:
result = await storage.search(query="freshness overlay two-read protocol", limit=5)
print(result.to_str())

/src/vfs/storage/backends/database/search.py  score=1.0000
/src/vfs/storage/backends/database/search.py:31-63
> sides share one formula, one accumulation order and one scale. The
> previews of the **overlay** entries that answer come from one bounded **read**
> of those entries' live bodies after the order is settled. The pointer
> **read** is advisory and re-issued after the fetch (the **two**-**read** **protocol**);
/src/vfs/storage/backends/database/search.py:986-1019
>     scored, the rest are absent, and the flag says so. Hits only come
>     back; the preview of an **overlay** entry that answers is a second **read**
>     of its live body (:func:`_**overlay**_bodies`), so a write landing
>     between the count and that **read** yields a preview from the newer body.
/src/vfs/storage/backends/database/search.py:272-309
>     Scoping arrives in grep's **two** shapes: pattern text on the ``globs``
>     channels below the router's composition limit, or past it the roots
>     beside

## 4. What a row carries

The same result as data. One `Observation` per entry; `score` is the entry's fused score (min-max scaled, so the best
hit is `1.0`); `matches` holds up to three chunk regions, best first, each with absolute 1-indexed line bounds, the raw
chunk text in `content`, and the preview with its own bounds — a sub-range of the region.

In [5]:
for row in result.observations[:3]:
    print(f"{row.path}  score={row.score:.4f}")
    for region in row.matches or []:
        print(f"  lines {region.start}-{region.end}  chunk score {region.score:.4f}")
        print(f"  preview lines {region.preview_start}-{region.preview_end}: {region.preview!r}")
        print(f"  raw text: {len(region.content or '')} chars")
    print()

/src/vfs/storage/backends/database/search.py  score=1.0000
  lines 31-63  chunk score 1.0000
  preview lines 40-43: "sides share one formula, one accumulation order and one scale. The\npreviews of the **overlay** entries that answer come from one bounded **read**\nof those entries' live bodies after the order is settled. The pointer\n**read** is advisory and re-issued after the fetch (the **two**-**read** **protocol**);"
  raw text: 1983 chars
  lines 986-1019  chunk score 0.3436
  preview lines 1009-1012: '    scored, the rest are absent, and the flag says so. Hits only come\n    back; the preview of an **overlay** entry that answers is a second **read**\n    of its live body (:func:`_**overlay**_bodies`), so a write landing\n    between the count and that **read** yields a preview from the newer body.'
  raw text: 1572 chars
  lines 272-309  chunk score 0.3388
  preview lines 274-277: "    Scoping arrives in grep's **two** shapes: pattern text on the ``globs``\n    channels below the

## 5. Projections

The renderer takes a projection — which columns to show. The native search vocabulary (`path`, `score`, `matches`)
prints as ranked lines; ask for a row-level field and it switches to a Markdown table, still in rank order.

In [6]:
print(result.to_str(projection=("path", "score")))
print()
print(result.to_str(projection=("path", "score", "size_bytes")))

/src/vfs/storage/backends/database/search.py  score=1.0000
/context/research/studies/2026-11-09-what-an-enterprise-scale-deployment-looks-like/03-indexing-cadence-full-and-incremental.md  score=0.9712
/context/research/2026-08-10-mcp-2026-07-28-stateless-revision.md  score=0.7758
/context/research/studies/2026-10-31-incremental-index-refresh/experiments/baseline/tables.md  score=0.6951
/context/research/2026-09-04-mcpfs-projecting-mcp-servers-into-a-namespace.md  score=0.6797

INFO: Unavailable: no embedding provider is configured; the lexical leg alone answered — Retry shortly; the resource is temporarily unavailable. (retry: transient)

| path                                                                                                                            |  score | size_bytes |
| ------------------------------------------------------------------------------------------------------------------------------- | -----: | ---------: |
| /src/vfs/storage/backends/database/search.p

## 6. Scope

Scoping is grep's: `globs`, `globs_not`, `ext`, `ext_not`, and `paths`. Whichever rung the ladder takes internally
(candidate ids for a narrow scope, score-then-gate for a wide one), a scoped call never returns an out-of-scope entry.

In [7]:
only_docs = await storage.search(query="block postings per term", limit=5, globs=("context/**",))
print(only_docs.to_str(projection=("path", "score")))
print()
only_code = await storage.search(query="block postings per term", limit=5, ext=("py",))
print(only_code.to_str(projection=("path", "score")))

/context/research/studies/2026-08-26-bm25-storage/vectorchord-bestmatch-lucene.md  score=1.0000
/context/research/studies/2026-08-26-bm25-storage/paradedb-tantivy.md  score=0.9289
/context/research/studies/2026-08-26-bm25-storage/pg_textsearch.md  score=0.8984
/context/research/studies/2026-08-26-bm25-storage/prototype-benchmark.md  score=0.8981
/context/research/studies/2026-08-26-bm25-storage/review-validation.md  score=0.8032

INFO: Unavailable: no embedding provider is configured; the lexical leg alone answered — Retry shortly; the resource is temporarily unavailable. (retry: transient)

/src/vfs/storage/backends/database/search.py  score=1.0000
/src/vfs/models/lexical.py  score=0.9733
/src/vfs/models/rows.py  score=0.9095
/src/vfs/storage/backends/database/lexical.py  score=0.8074
/src/vfs/storage/backends/database/pathterms.py  score=0.4362

INFO: Unavailable: no embedding provider is configured; the lexical leg alone answered — Retry shortly; the resource is temporarily unavaila

## 7. The freshness overlay

A file written after the last rebuild is not in the lexical index yet. Search still finds it: unindexed entries are
scored from their live text with the index's own statistics, so the two sides share one scale. An overlay hit answers
as one whole document — its region spans the file, carries no chunk text, and its preview is cut from the body that was
already read to score it (no second fetch).

In [8]:
fresh = Entry(
    path=Path("/notes/lantern.md"),
    content="# Lantern protocol\n\nA lantern is a signal the router raises when a mount goes dark.\n"
    "Lanterns are cheap; the light is the point.\n",
)
await storage.write(entries=[fresh], parents=True)

before = await storage.search(query="lantern signal", limit=3)
print("before rebuild:")
print(before.to_str())
hit = before.observations[0]  # the fresh file outranks the memos that mention lanterns
[region] = hit.matches or []
print("\nregion bounds:", (region.start, region.end), " content carried:", region.content is not None)

await storage.rebuild()
after = await storage.search(query="lantern signal", limit=3)
print("\nafter rebuild:")
print(after.to_str())

before rebuild:
/notes/lantern.md  score=1.0000
/notes/lantern.md:1-4
> # **Lantern** protocol
>
> A **lantern** is a **signal** the router raises when a mount goes dark.
> **Lantern**s are cheap; the light is the point.
/context/research/studies/2026-08-26-glean/engine-matrix.md  score=0.3472
/context/research/studies/2026-08-26-glean/engine-matrix.md:133-138
> …bucketed into 32 ranges (docs). `CONTAINSTABLE(..., '**lantern**')` returned `RANK = 192` for the tf=3 chunks. Tokenizer control is the per-column `LANGUAGE` word b…
>
> ### 2.5 Oracle 26ai Free 23.26.2 (the `23` tags) — regular image
/context/research/studies/2026-08-26-glean/engine-matrix.md:13-29
>
> ## 1. The fused statement: one portable shape, verified on five engines
>
> …× 4 chunks, 8-dim synthetic vectors, lexical target "**lantern**" planted in entries 30–39, scope `src/mod3/**` = 10 entry ids, `k_rrf=60`, per-leg `k=20`, `n=5`):
/src/vfs/storage/backends/database/signals.py  score=0.2505
/src/vfs/storage/backends/da


after rebuild:
/notes/lantern.md  score=1.0000
/notes/lantern.md:1-4
> # **Lantern** protocol
>
> A **lantern** is a **signal** the router raises when a mount goes dark.
> **Lantern**s are cheap; the light is the point.
/context/research/studies/2026-08-26-glean/engine-matrix.md  score=0.3377
/context/research/studies/2026-08-26-glean/engine-matrix.md:133-138
> …bucketed into 32 ranges (docs). `CONTAINSTABLE(..., '**lantern**')` returned `RANK = 192` for the tf=3 chunks. Tokenizer control is the per-column `LANGUAGE` word b…
>
> ### 2.5 Oracle 26ai Free 23.26.2 (the `23` tags) — regular image
/context/research/studies/2026-08-26-glean/engine-matrix.md:13-29
>
> ## 1. The fused statement: one portable shape, verified on five engines
>
> …× 4 chunks, 8-dim synthetic vectors, lexical target "**lantern**" planted in entries 30–39, scope `src/mod3/**` = 10 entry ids, `k_rrf=60`, per-leg `k=20`, `n=5`):
/src/vfs/storage/backends/database/signals.py  score=0.2594
/src/vfs/storage/backends/da

## 8. The statistics behind the ranking

Every search result carries a `lexical_stats` extra: the corpus size, the mean document length, and each query term's
document frequency and idf. A term missing here occurred nowhere in the corpus. Note that identifiers split —
`line_start` becomes `line_start`, `line`, and `start` — which is also why the bolder marks short terms inside longer words.

In [9]:
stats = (await storage.search(query="chunk line_start embedding", limit=1)).lexical_stats
print("documents:", stats["n_docs"], " mean length:", round(stats["avg_dl"], 1))
for term, facts in stats["terms"].items():
    print(f"  {term:12s} df={facts['df']:4d}  idf={facts['idf']:.3f}")

documents: 8211  mean length: 240.0
  chunk        df=1016  idf=2.089
  line_start   df=  26  idf=5.736
  line         df=1196  idf=1.926
  start        df= 327  idf=3.222
  embedding    df= 304  idf=3.295


## 9. The preview selector on its own

`select_preview` is a pure function: chunk text, the chunk's first line number, the query's folded terms. It folds the
chunk once, scores each line by the distinct terms it holds (whole-word hits count double a substring hit), slides a
4-line window, bolds merged spans, and trims to 160 characters a line and 480 a preview. No term in the chunk → the head
of the chunk, unbolded. The display budgets are keyword arguments, never search parameters.

In [10]:
chunk = (root / "src/vfs/results/preview.py").read_text()
excerpt = select_preview(chunk, 1, ["fold", "window", "cap"])
print(f"lines {excerpt.start}-{excerpt.end}, matched={excerpt.matched}")
print(excerpt.text)
print()
narrow = select_preview(chunk, 1, ["fold", "window", "cap"], window=2, line_cap=60)
print(f"lines {narrow.start}-{narrow.end}")
print(narrow.text)
print()
head = select_preview(chunk, 1, ["zzqx"])
print(f"no hit -> head window, lines {head.start}-{head.end}, matched={head.matched}")

lines 98-101, matched=True
    **cap**ped bonus per extra occurrence and a bonus for adjacent distinct
    terms. A **window** scores the sum of its lines plus a bonus per distinct
    term it covers; the best **window** wins, the earliest on ties. With no
    term anywhere in the chunk, the head **window** comes back unbolded.

lines 4-5
text alone: one **fold** of the chunk, a score per line from the…
terms, the best short **window** of lines, the terms bolded in M…

no hit -> head window, lines 1-4, matched=False


## 10. Through the router, across mounts

An `AsyncVirtualFileSystem` routes one namespace over many stores. Mount two backends and search from the root: each mount
answers in its own coordinates and the router rebases the paths on the way back. (The cross-mount *rerank* is a later spec; today each mount's scores are on its own scale, so two mounts can both
show a `1.0000`.)

In [11]:
code_store = DatabaseStorage(url=f"sqlite+aiosqlite:///{workdir}/code.sqlite")
docs_store = DatabaseStorage(url=f"sqlite+aiosqlite:///{workdir}/docs.sqlite")
await code_store.write(entries=entries_under("src/vfs", "*.py"), parents=True)
await docs_store.write(entries=entries_under("context/research", "*.md"), parents=True)
await code_store.rebuild()
await docs_store.rebuild()

fs = AsyncVirtualFileSystem()
await fs.add_mount(code_store, "/code")
await fs.add_mount(docs_store, "/docs")

merged = await fs.search("two-read protocol epoch pointer", limit=6)
print(merged.to_str(projection=("path", "score")))
print()
scoped = await fs.search("two-read protocol epoch pointer", limit=3, paths=("/docs",))
print(scoped.to_str(projection=("path", "score")))

/docs/context/research/studies/2026-10-31-incremental-index-refresh/04-design-options.md  score=1.0000
/docs/context/research/2026-08-13-glob-grep-indexing-review-campaign.md  score=0.9047
/code/src/vfs/storage/backends/database/search.py  score=0.8346
/docs/context/research/2026-10-31-incremental-index-refresh-measured-and-the-delta-segment-recommendation.md  score=0.8320
/code/src/vfs/storage/backends/database/reads.py  score=0.8103
/code/src/vfs/storage/backends/database/grep.py  score=0.7851

INFO mount /code: Unavailable: no embedding provider is configured; the lexical leg alone answered — Retry shortly; the resource is temporarily unavailable. (retry: transient)
INFO mount /docs: Unavailable: no embedding provider is configured; the lexical leg alone answered — Retry shortly; the resource is temporarily unavailable. (retry: transient)

/docs/context/research/studies/2026-10-31-incremental-index-refresh/04-design-options.md  score=1.0000
/docs/context/research/2026-08-13-glob-gre

## 11. Clean up

In [12]:
await fs.close()
await storage.close()
print("closed")

closed
